# Laya Model Training for WCAG 2.2 Violation Detection

Fine-tunes the Laya model to detect WCAG 2.2 accessibility violations.
Output: ONNX model compatible with layaForWeb.

In [ ]:
!pip install -q torch transformers datasets onnx onnxruntime laya

## Load WCAG 2.2 Training Data

Option A: Use existing WCAG violation datasets
Option B: Generate synthetic data using LLM
Option C: Combine multiple sources

In [ ]:
# Setup
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HFDataset

# Define WCAG 2.2 criteria
WCAG_22_CRITERIA = [
    {"criterion": "1.1.1", "principle": "Perceivable", "level": "A", "description": "Non-text Content"},
    {"criterion": "1.2.1", "principle": "Perceivable", "level": "A", "description": "Audio-only and Video-only"},
    {"criterion": "1.2.2", "principle": "Perceivable", "level": "A", "description": "Captions (Prerecorded)"},
    {"criterion": "1.3.1", "principle": "Perceivable", "level": "A", "description": "Info and Relationships"},
    {"criterion": "1.4.1", "principle": "Perceivable", "level": "A", "description": "Use of Color"},
]
print(f"Loaded {len(WCAG_22_CRITERIA)} WCAG criteria")


In [ ]:
def generate_synthetic_training_data(num_samples=1000):
    """Generate synthetic WCAG violation training data."""
    training_data = []
    
    for i in range(num_samples):
        text = f"Web page {i}: Review for WCAG 2.2 accessibility compliance."
        # Randomly assign violations
        violations = []
        for criterion in WCAG_22_CRITERIA:
            if i % (hash(criterion['criterion']) % 7 + 1) == 0:
                violations.append(criterion)
        
        training_data.append({
            "text": text,
            "violations": violations
        })
    
    return training_data

training_data = generate_synthetic_training_data(500)
print(f"Generated {len(training_data)} synthetic training samples")

In [ ]:
from datasets import Dataset

def create_hf_dataset(data):
    """Convert list of dicts to HuggingFace Dataset."""
    texts = [item["text"] for item in data]
    violation_lists = [item["violations"] for item in data]
    
    return Dataset.from_dict({
        "text": texts,
        "violations": violation_lists
    })

hf_dataset = create_hf_dataset(training_data)
train_test_split = hf_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
val_dataset = train_test_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
## Load Base Laya Model

import laya

# Use laya.load which handles the internal encoder/tokenizer structure
agent = laya.load("convaiinnovations/laya", device=device)
base_model = agent.model

print(f"Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
# Use laya's tokenizer from the loaded agent
tokenizer = agent.tok

def tokenize_function(examples):
    """Tokenize text inputs for the model."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("Tokenization complete")

In [ ]:
## Multi-Label Classification Head

class LayaWCAGModel(nn.Module):
    def __init__(self, base_model, num_criteria):
        super().__init__()
        self.base = base_model
        # DecisionModel stores encoder as self.encoder; config is on the encoder
        self.classifier = nn.Linear(base_model.encoder.config.hidden_size, num_criteria)
    
    def forward(self, input_ids, attention_mask):
        # Access encoder directly; avoid DecisionModel.forward signature mismatch
        h = self.base.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = h[:, 0].float()  # CLS token
        logits = self.classifier(pooled)
        return logits

num_criteria = len(WCAG_22_CRITERIA)
model = LayaWCAGModel(base_model, num_criteria)
model.to(device)

print(f"Model initialized with {num_criteria} criteria classification head")

In [ ]:
## Training Configuration

# Training hyperparameters
learning_rate = 2e-5
batch_size = 8
num_epochs = 3
num_criteria = len(WCAG_22_CRITERIA)

# Create data collator
from torch.utils.data import DataLoader

# Simple training arguments
print(f"Training config:")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {num_epochs}")
print(f"  Criteria: {num_criteria}")

In [ ]:
## DataLoader with Label Encoding

def create_label_map(criterias):
    """Create a mapping from criterion IDs to indices."""
    return {c['criterion']: i for i, c in enumerate(criterias)}

label_map = create_label_map(WCAG_22_CRITERIA)

def encode_labels(examples):
    """Convert violations list to multi-hot encoded labels."""
    labels = []
    for violation_list in examples['violations']:
        label = [0] * num_criteria
        for v in violation_list:
            idx = label_map.get(v['criterion'])
            if idx is not None:
                label[idx] = 1
        labels.append(label)
    return {'labels': labels}

# Add labels column to datasets
train_dataset_with_labels = train_dataset.map(encode_labels, batched=True)
val_dataset_with_labels = val_dataset.map(encode_labels, batched=True)

train_dataset_with_labels.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset_with_labels.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_loader = DataLoader(train_dataset_with_labels, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset_with_labels, batch_size=batch_size, shuffle=False)

print(f"Created {len(train_loader)} training batches, {len(val_loader)} validation batches")

In [ ]:
## Training Loop

import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Optimizer
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

criterion = nn.BCEWithLogitsLoss()

print("Model ready for training")
print(f"Total training steps: {total_steps}")

In [ ]:
for batch in train_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    # Labels are in the violations column (list of dicts)
    labels = []
    for v in batch['violations']:
        label = [0] * num_criteria
        for crit in v:
            idx = label_map.get(crit['criterion'])
            if idx is not None:
                label[idx] = 1
        labels.append(label)
    labels = torch.tensor(labels, dtype=torch.float32).to(device)
    
    optimizer.zero_grad()
    logits = model(input_ids, attention_mask)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()
    total_loss += loss.item()

In [ ]:
## Model Evaluation

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = model(input_ids, attention_mask)
        preds = (logits > 0).int().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

macro_f1 = f1_score(all_labels, all_preds, average='macro')
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')

print(f"Validation Results:")
print(f"  Macro F1: {macro_f1:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")

In [ ]:
## ONNX Export and Quantization

import torch.onnx
import onnx

# Create dummy inputs
dummy_input_ids = torch.ones(1, 512, dtype=torch.long).to(device)
dummy_attention_mask = torch.ones(1, 512, dtype=torch.long).to(device)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    "laya-wcag.onnx",
    opset_version=17,
    input_names=['input_ids', 'attention_mask'],
    output_names=['criterion_scores', 'confidence'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'criterion_scores': {0: 'batch_size'}
    }
)

# Verify ONNX model
onnx_model = onnx.load("laya-wcag.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX export successful")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
## Quantization and Model Packaging

import os
import hashlib
import json as json_lib

# Create output directory
os.makedirs('build/laya-wcag', exist_ok=True)

# Split model into 24MiB parts
chunk_size = 24 * 1024 * 1024  # 24 MiB
manifest = []

with open('laya-wcag.onnx', 'rb') as f:
    part_num = 0
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        
        filename = f'laya-wcag.{part_num:04d}.onnx.data'
        with open(f'build/laya-wcag/{filename}', 'wb') as out:
            out.write(chunk)
        
        # Calculate SHA-256
        sha256 = hashlib.sha256(chunk).hexdigest()
        manifest.append({
            'filename': filename,
            'size': len(chunk),
            'sha256': sha256
        })
        part_num += 1

# Save manifest
manifest_path = 'build/laya-wcag/manifest.json'
with open(manifest_path, 'w') as f:
    json_lib.dump(manifest, f, indent=2)

print(f"Model packaged: {part_num} parts, {len(manifest_path)} manifest entries")
print("Ready for deployment")